# Insurance Claims Data Quality Audit
**Author:** Aryaa  
**Dataset:** General Insurance Claims | Jan 2023 – Jun 2024  
**Tools:** Python (pandas), SQL (SQLite), Excel

---

The goal of this project is to check the quality of an insurance claims dataset before it gets used for finance reporting. In a real team, bad data reaching accountants or actuaries can cause wrong reserve calculations, incorrect reporting, and in the worst case, financial losses from overpayments.

I've structured the checks using a standard DQ framework:

| Category | What I'm checking |
|---|---|
| Completeness | Are required fields filled in? |
| Uniqueness | Are there any duplicate records? |
| Validity | Do values make sense / follow the rules? |
| Consistency | Are related fields logically consistent? |
| Integrity | Do financial figures follow business rules? |


## Setup — import libraries

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# just making output a bit cleaner to read
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)


## Load the data

In [2]:
df = pd.read_csv("insurance_claims_raw.csv")

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print()
print(df.dtypes)


Rows: 508
Columns: 12

claim_id                object
policy_number           object
customer_id             object
claim_type              object
incident_date           object
report_date             object
claim_amount_gbp       float64
approved_amount_gbp    float64
status                  object
region                  object
assessor_id             object
fraud_flag               int64
dtype: object


In [3]:
# quick look at the first few rows
df.head()


,claim_id,policy_number,customer_id,claim_type,incident_date,report_date,claim_amount_gbp,approved_amount_gbp,status,region,assessor_id,fraud_flag
0,CLM-00080,POL-403445,CUST-55309,Property,2023-12-10,2023-12-21,618.38,480.02,Closed,Central,A018,1
1,CLM-00317,POL-522179,CUST-70952,Property,2023-09-12,2023-10-07,"1,559.13",NaN,Closed,Central,A013,0
2,CLM-00486,POL-987204,CUST-85631,PropErty,2024-03-18,2024-03-22,264.36,170.52,Open,East,A007,0
3,CLM-00397,POL-723939,CUST-46655,PropErty,2024-01-21,2024-02-08,725.45,472.97,Open,East,A014,0
4,CLM-00168,POL-166613,CUST-85454,Health,2024-01-14,2024-01-14,"1,911.60","1,211.57",Open,North,A002,0


In [4]:
# check numeric columns
df.describe()


,claim_amount_gbp,approved_amount_gbp,fraud_flag
count,508.00,431.00,508.00
mean,"4,253.97","3,408.95",0.07
std,"4,661.35","4,139.75",0.26
min,"-11,523.61","-21,082.95",0.00
25%,"1,138.99",933.85,0.00
50%,"3,006.43","2,470.80",0.00
75%,"5,961.34","4,438.09",0.00
max,"28,824.63","30,770.07",1.00


## Load into SQLite so I can run SQL queries

I want to show the checks both in Python and SQL — in most finance teams you'd be querying a data warehouse with SQL, so it's good to have both.


In [5]:
conn = sqlite3.connect(":memory:")
df.to_sql("insurance_claims", conn, if_exists="replace", index=False)

# just checking it loaded correctly
pd.read_sql("SELECT COUNT(*) AS total FROM insurance_claims", conn)


,total
0,508


## DQ-01 | Completeness — Missing values

Starting with nulls because this is usually the most common issue. Missing customer IDs or assessor IDs would break any downstream reporting or audit trail.


In [6]:
# which columns have nulls?
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

missing = pd.DataFrame({
    "null_count": null_counts,
    "null_%": null_pct
})

# only show columns that actually have nulls
missing[missing["null_count"] > 0].sort_values("null_count", ascending=False)


,null_count,null_%
approved_amount_gbp,77,15.16
customer_id,18,3.54
region,13,2.56
assessor_id,10,1.97
claim_type,8,1.57


In [7]:
# same thing in SQL
sql = '''
SELECT
    'customer_id' AS field,
    COUNT(*) FILTER (WHERE customer_id IS NULL) AS null_count,
    ROUND(COUNT(*) FILTER (WHERE customer_id IS NULL) * 100.0 / COUNT(*), 2) AS null_pct
FROM insurance_claims
UNION ALL
SELECT 'region',
    COUNT(*) FILTER (WHERE region IS NULL),
    ROUND(COUNT(*) FILTER (WHERE region IS NULL) * 100.0 / COUNT(*), 2)
FROM insurance_claims
UNION ALL
SELECT 'assessor_id',
    COUNT(*) FILTER (WHERE assessor_id IS NULL),
    ROUND(COUNT(*) FILTER (WHERE assessor_id IS NULL) * 100.0 / COUNT(*), 2)
FROM insurance_claims
UNION ALL
SELECT 'claim_type',
    COUNT(*) FILTER (WHERE claim_type IS NULL),
    ROUND(COUNT(*) FILTER (WHERE claim_type IS NULL) * 100.0 / COUNT(*), 2)
FROM insurance_claims
'''

pd.read_sql(sql, conn)


,field,null_count,null_pct
0,customer_id,18,3.54
1,region,13,2.56
2,assessor_id,10,1.97
3,claim_type,8,1.57


## DQ-02 | Uniqueness — Duplicate claim IDs

Each claim should have a unique ID. Duplicates could mean the same claim gets paid out twice, which is a direct financial risk.


In [8]:
dupes = df[df.duplicated(subset=["claim_id"], keep=False)].sort_values("claim_id")

print(f"Duplicate records found: {df.duplicated(subset=['claim_id']).sum()}")
print()
dupes[["claim_id", "policy_number", "claim_amount_gbp", "status"]].head(10)


Duplicate records found: 8



,claim_id,policy_number,claim_amount_gbp,status
128,CLM-00026,POL-308496,900.93,Open
465,CLM-00026,POL-308496,900.93,Open
237,CLM-00060,POL-865179,"-6,008.05",Closed
499,CLM-00060,POL-865179,"6,008.05",Closed
92,CLM-00102,POL-765822,"1,822.36",Open
452,CLM-00102,POL-765822,"1,822.36",Open
159,CLM-00160,POL-214975,"9,114.22",Open
334,CLM-00160,POL-214975,"9,114.22",Open
68,CLM-00182,POL-887352,"2,061.58",Closed
84,CLM-00182,POL-887352,"2,061.58",Closed


In [9]:
# SQL version
sql = '''
SELECT claim_id, COUNT(*) AS times_seen
FROM insurance_claims
GROUP BY claim_id
HAVING COUNT(*) > 1
ORDER BY times_seen DESC
'''

pd.read_sql(sql, conn)


,claim_id,times_seen
0,CLM-00444,2
1,CLM-00398,2
2,CLM-00327,2
3,CLM-00182,2
4,CLM-00160,2
5,CLM-00102,2
6,CLM-00060,2
7,CLM-00026,2


## DQ-03 & DQ-04 | Validity — Claim amounts that don't make sense

Negative amounts suggest something went wrong in data entry or the source system. Zero amounts could be legit (e.g. a withdrawn claim) but they need to be reviewed rather than just accepted.


In [10]:
negative_claims = df[df["claim_amount_gbp"] < 0]
zero_claims = df[df["claim_amount_gbp"] == 0]

print(f"Negative amounts: {len(negative_claims)} records")
print(f"Zero amounts: {len(zero_claims)} records")


Negative amounts: 7 records
Zero amounts: 6 records


In [11]:
negative_claims[["claim_id", "claim_type", "claim_amount_gbp", "status"]]


,claim_id,claim_type,claim_amount_gbp,status
52,CLM-00212,Property,"-5,301.81",Open
76,CLM-00040,motor,"-7,696.36",Open
172,CLM-00174,Property,"-11,523.61",Closed
237,CLM-00060,Motor,"-6,008.05",Closed
280,CLM-00435,Liability,-717.87,Closed
404,CLM-00237,Property,-663.60,Closed
476,CLM-00460,Liability,"-1,435.17",Closed


In [12]:
# SQL — checking both in one go
sql = '''
SELECT claim_id, policy_number, claim_amount_gbp,
    CASE
        WHEN claim_amount_gbp < 0 THEN 'Negative'
        WHEN claim_amount_gbp = 0 THEN 'Zero'
    END AS issue
FROM insurance_claims
WHERE claim_amount_gbp <= 0
ORDER BY claim_amount_gbp
'''

pd.read_sql(sql, conn)


,claim_id,policy_number,claim_amount_gbp,issue
0,CLM-00174,POL-363626,"-11,523.61",Negative
1,CLM-00040,POL-267414,"-7,696.36",Negative
2,CLM-00060,POL-865179,"-6,008.05",Negative
3,CLM-00212,POL-665579,"-5,301.81",Negative
4,CLM-00460,POL-956795,"-1,435.17",Negative
5,CLM-00435,POL-690341,-717.87,Negative
6,CLM-00237,POL-903035,-663.60,Negative
7,CLM-00404,POL-358175,0.00,Zero
8,CLM-00114,POL-330283,0.00,Zero
9,CLM-00011,POL-809570,0.00,Zero


## DQ-05 | Consistency — Report date before incident date

This one's a logic check. You can't report a claim before the incident happened. These records have a report_date that's earlier than the incident_date, which is impossible.


In [13]:
# convert to datetime first so we can compare properly
df["incident_date_dt"] = pd.to_datetime(df["incident_date"], errors="coerce")
df["report_date_dt"]   = pd.to_datetime(df["report_date"], errors="coerce")

bad_dates = df[df["report_date_dt"] < df["incident_date_dt"]].copy()
bad_dates["days_out"] = (bad_dates["incident_date_dt"] - bad_dates["report_date_dt"]).dt.days

print(f"Records with report date before incident date: {len(bad_dates)}")


Records with report date before incident date: 19


In [14]:
bad_dates[["claim_id", "incident_date", "report_date", "days_out", "status"]].sort_values(
    "days_out", ascending=False).head(8)


,claim_id,incident_date,report_date,days_out,status
435,CLM-00264,2026-12-03,2023-08-02,1219,Pending
442,CLM-00346,2026-12-03,2023-11-29,1100,Pending
291,CLM-00383,2026-12-03,2024-04-03,974,Rejected
416,CLM-00162,2026-12-03,2024-05-26,921,Open
370,CLM-00033,2026-12-03,2024-06-30,886,Closed
296,CLM-00372,2023-12-12,2023-11-26,16,Open
266,CLM-00097,2023-09-30,2023-09-17,13,Closed
384,CLM-00063,2023-06-11,2023-05-29,13,Pending


In [15]:
# SQL check
sql = '''
SELECT claim_id, incident_date, report_date,
    CAST(JULIANDAY(incident_date) - JULIANDAY(report_date) AS INTEGER) AS days_out
FROM insurance_claims
WHERE report_date < incident_date
ORDER BY days_out DESC
LIMIT 10
'''

pd.read_sql(sql, conn)


,claim_id,incident_date,report_date,days_out
0,CLM-00264,2026-12-03,2023-08-02,1219
1,CLM-00346,2026-12-03,2023-11-29,1100
2,CLM-00383,2026-12-03,2024-04-03,974
3,CLM-00162,2026-12-03,2024-05-26,921
4,CLM-00033,2026-12-03,2024-06-30,886
5,CLM-00372,2023-12-12,2023-11-26,16
6,CLM-00097,2023-09-30,2023-09-17,13
7,CLM-00063,2023-06-11,2023-05-29,13
8,CLM-00183,2023-06-17,2023-06-08,9
9,CLM-00067,2024-04-22,2024-04-13,9


## DQ-06 | Validity — Future incident dates

An incident can't happen in the future. These are almost certainly data entry mistakes — someone typed the wrong year, or there's a system clock issue.


In [16]:
today = pd.Timestamp.today()
future_records = df[df["incident_date_dt"] > today]

print(f"Records with future incident dates: {len(future_records)}")
future_records[["claim_id", "incident_date", "claim_type", "claim_amount_gbp"]]


Records with future incident dates: 5


,claim_id,incident_date,claim_type,claim_amount_gbp
291,CLM-00383,2026-12-03,Liability,"5,943.48"
370,CLM-00033,2026-12-03,Property,"2,973.46"
416,CLM-00162,2026-12-03,Property,"7,023.57"
435,CLM-00264,2026-12-03,Motor,"5,243.33"
442,CLM-00346,2026-12-03,NaN,"8,303.18"


## DQ-07 | Validity — Status values that aren't in the expected list

The status field should only contain: Open, Closed, Pending, Rejected. Anything else is either a typo or a system error. These invalid values would break any filters or reports that group by status.


In [17]:
valid_statuses = {"Open", "Closed", "Pending", "Rejected"}
bad_status = df[~df["status"].isin(valid_statuses)]

print(f"Records with invalid status: {len(bad_status)}")
print()
print(bad_status["status"].value_counts())


Records with invalid status: 10

status
Clsoed     2
closed     1
open       1
pENDING    1
PNDNG      1
OPEN       1
Rejectd    1
Pendng     1
CLOSED     1
Name: count, dtype: int64


In [18]:
# SQL
sql = '''
SELECT status, COUNT(*) AS count
FROM insurance_claims
WHERE status NOT IN ('Open', 'Closed', 'Pending', 'Rejected')
GROUP BY status
ORDER BY count DESC
'''

pd.read_sql(sql, conn)


,status,count
0,Clsoed,2
1,pENDING,1
2,open,1
3,closed,1
4,Rejectd,1
5,Pendng,1
6,PNDNG,1
7,OPEN,1
8,CLOSED,1


## DQ-09 | Integrity — Approved amount higher than claim amount

This is the most financially serious issue in the dataset. If someone is paid out more than they claimed, that's a direct financial loss. Every one of these records needs to go to the finance team for review.


In [19]:
overpaid = df[
    df["approved_amount_gbp"].notna() &
    (df["approved_amount_gbp"] > df["claim_amount_gbp"]) &
    (df["claim_amount_gbp"] > 0)
].copy()

overpaid["extra_paid"] = (overpaid["approved_amount_gbp"] - overpaid["claim_amount_gbp"]).round(2)

print(f"Claims where approved > claimed: {len(overpaid)}")
print(f"Total overpayment exposure: £{overpaid['extra_paid'].sum():,.2f}")


Claims where approved > claimed: 15
Total overpayment exposure: £36,740.53


In [20]:
overpaid[["claim_id", "claim_amount_gbp", "approved_amount_gbp", "extra_paid"]].sort_values(
    "extra_paid", ascending=False).head(10)


,claim_id,claim_amount_gbp,approved_amount_gbp,extra_paid
153,CLM-00318,"17,597.43","30,770.07","13,172.64"
70,CLM-00133,"6,334.21","9,980.98","3,646.77"
337,CLM-00029,508.64,"3,928.49","3,419.85"
130,CLM-00043,"1,072.41","3,980.63","2,908.22"
6,CLM-00064,200.00,"3,028.92","2,828.92"
154,CLM-00480,"4,239.84","6,601.89","2,362.05"
205,CLM-00093,"3,294.06","5,591.38","2,297.32"
408,CLM-00465,"2,526.83","4,812.83","2,286.00"
393,CLM-00015,"1,652.15","2,561.29",909.14
328,CLM-00066,"4,203.83","5,053.21",849.38


In [21]:
# SQL
sql = '''
SELECT claim_id, claim_amount_gbp, approved_amount_gbp,
    ROUND(approved_amount_gbp - claim_amount_gbp, 2) AS extra_paid
FROM insurance_claims
WHERE approved_amount_gbp > claim_amount_gbp
  AND claim_amount_gbp > 0
  AND approved_amount_gbp IS NOT NULL
ORDER BY extra_paid DESC
LIMIT 10
'''

pd.read_sql(sql, conn)


,claim_id,claim_amount_gbp,approved_amount_gbp,extra_paid
0,CLM-00318,"17,597.43","30,770.07","13,172.64"
1,CLM-00133,"6,334.21","9,980.98","3,646.77"
2,CLM-00029,508.64,"3,928.49","3,419.85"
3,CLM-00043,"1,072.41","3,980.63","2,908.22"
4,CLM-00064,200.00,"3,028.92","2,828.92"
5,CLM-00480,"4,239.84","6,601.89","2,362.05"
6,CLM-00093,"3,294.06","5,591.38","2,297.32"
7,CLM-00465,"2,526.83","4,812.83","2,286.00"
8,CLM-00015,"1,652.15","2,561.29",909.14
9,CLM-00066,"4,203.83","5,053.21",849.38


## DQ-10 | Consistency — Rejected claims that still have an approved payout

If a claim is rejected, there shouldn't be any approved amount attached to it. These records contradict basic business logic and need to be escalated — they could also be a fraud indicator.


In [22]:
rejected_with_payout = df[
    (df["status"] == "Rejected") &
    df["approved_amount_gbp"].notna()
]

print(f"Rejected claims with a payout amount: {len(rejected_with_payout)}")
print(f"Total value: £{rejected_with_payout['approved_amount_gbp'].sum():,.2f}")


Rejected claims with a payout amount: 36
Total value: £154,620.03


In [23]:
rejected_with_payout[["claim_id", "status", "claim_amount_gbp", "approved_amount_gbp", "region"]].head(10)


,claim_id,status,claim_amount_gbp,approved_amount_gbp,region
5,CLM-00494,Rejected,"2,688.14","2,561.91",North
6,CLM-00064,Rejected,200.00,"3,028.92",South
38,CLM-00154,Rejected,"1,052.27",598.01,Central
55,CLM-00079,Rejected,"21,327.24","19,236.73",North
78,CLM-00047,Rejected,"10,362.67","4,438.04",South
79,CLM-00395,Rejected,"2,064.47","1,265.65",East
97,CLM-00500,Rejected,"10,661.06","6,965.81",South
108,CLM-00352,Rejected,"18,800.67","13,504.65",North
130,CLM-00043,Rejected,"1,072.41","3,980.63",West
143,CLM-00025,Rejected,"5,143.20","1,265.59",West


## Summary — DQ scorecard

In [24]:
# pull together a summary of all checks
checks_summary = {
    "DQ-01 Missing values":         {"affected": int(df.isnull().sum().sum()),         "severity": "HIGH"},
    "DQ-02 Duplicate claim IDs":    {"affected": int(df.duplicated("claim_id").sum()), "severity": "HIGH"},
    "DQ-03 Negative amounts":       {"affected": int(len(df[df["claim_amount_gbp"] < 0])), "severity": "HIGH"},
    "DQ-04 Zero amounts":           {"affected": int(len(df[df["claim_amount_gbp"] == 0])), "severity": "MEDIUM"},
    "DQ-05 Date logic error":       {"affected": int(len(bad_dates)),                  "severity": "HIGH"},
    "DQ-06 Future dates":           {"affected": int(len(future_records)),             "severity": "HIGH"},
    "DQ-07 Invalid status values":  {"affected": int(len(bad_status)),                "severity": "HIGH"},
    "DQ-09 Overpayment":            {"affected": int(len(overpaid)),                  "severity": "CRITICAL"},
    "DQ-10 Rejected + payout":      {"affected": int(len(rejected_with_payout)),      "severity": "CRITICAL"},
}

scorecard = pd.DataFrame(checks_summary).T.reset_index()
scorecard.columns = ["Check", "Records Affected", "Severity"]

print(scorecard.to_string(index=False))

# rough DQ score
total_cells = len(df) * len(df.columns)
weights = {"CRITICAL": 3, "HIGH": 2, "MEDIUM": 1}
scorecard["weight"] = scorecard["Severity"].map(weights)
scorecard["weighted"] = scorecard["Records Affected"].astype(int) * scorecard["weight"]

score = round(100 - (scorecard["weighted"].sum() / total_cells * 100), 1)
print(f"\nOverall DQ Score: {score} / 100")


                      Check Records Affected Severity
       DQ-01 Missing values              126     HIGH
  DQ-02 Duplicate claim IDs                8     HIGH
     DQ-03 Negative amounts                7     HIGH
         DQ-04 Zero amounts                6   MEDIUM
     DQ-05 Date logic error               19     HIGH
         DQ-06 Future dates                5     HIGH
DQ-07 Invalid status values               10     HIGH
          DQ-09 Overpayment               15 CRITICAL
    DQ-10 Rejected + payout               36 CRITICAL

Overall DQ Score: 92.8 / 100


## Next steps

Based on the findings, here's what I'd recommend:

**Immediate (this week):**
- Escalate DQ-09 and DQ-10 to the finance team — these are financial integrity issues
- DQ-02 duplicates need to be removed and the root cause found in the pipeline

**Short term (next few weeks):**
- Add validation rules at the point of data entry: block negative amounts, future dates, invalid status values
- Enforce a primary key on claim_id so duplicates can't get through

**Ongoing:**
- Run these checks automatically at each monthly close
- Track the DQ score over time so any deterioration gets spotted early
